In [326]:
from datasets import load_from_disk

dataset = load_from_disk("combined_dataset")

print(dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x1280 at 0x33141E110>, 'caption': 'a yellow and white cartoon character with a red eye'}


In [327]:
image = dataset[0]["image"]
text = dataset[0]["caption"]

print(text)
image.show()

a yellow and white cartoon character with a red eye


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [328]:
full_dataset = dataset
print(len(full_dataset))


100


In [329]:
image_list = [item["image"].convert("RGB") for item in full_dataset]
caption_list = [item["caption"] for item in full_dataset]

In [330]:
# from datasets import Dataset
# import random

# # Set seed for reproducibility (optional but recommended)
# seed = 42

# # Shuffle and select 20k samples
# train_dataset = full_dataset.shuffle(seed=seed).select(range(20000))

# print(train_dataset)




from datasets import Dataset

# Set seed for reproducibility
seed = 42

# Step 1: Shuffle full dataset
shuffled_dataset = full_dataset.shuffle(seed=seed)

# Step 2: Select splits
val_dataset = shuffled_dataset.select(range(20))
train_dataset = shuffled_dataset.select(range(20, 100))

# Print info

print("Validation size:", len(val_dataset))
print("Training size:", len(train_dataset))

Validation size: 20
Training size: 80


In [331]:
import random
final_val_pairs=val_dataset


print("Final validation dataset size:", len(final_val_pairs))

Final validation dataset size: 20


In [332]:
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
import random
import os

from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset

In [333]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

state_dict = torch.load("/Users/ssingodia/Desktop/Project-3/Task_5/best_clip_model.pt", map_location=device)
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Model loaded successfully ✅")

/Users/ssingodia/Desktop/Project-3/clipenv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded successfully ✅


In [334]:
# Extract images and captions
image_list = []
caption_list = []

for img, cap in final_val_pairs:
    image_list.append(img)
    caption_list.append(cap)

print("Images:", len(image_list))
print("Captions:", len(caption_list))

Images: 20
Captions: 20


In [335]:
# from torch.utils.data import DataLoader

In [336]:
# from torch.utils.data import Dataset

In [337]:
# class FlickrDataset(Dataset):
#     def __init__(self, pairs):
#         self.pairs = pairs
        
#     def __len__(self):
#         return len(self.pairs)
    
#     def __getitem__(self, idx):
#         image, caption = self.pairs[idx]
#         return image, caption


In [338]:
# from transformers import CLIPProcessor

# processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# def collate_fn(batch):
#     images, captions = zip(*batch)
    
#     inputs = processor(
#         text=list(captions),
#         images=list(images),
#         return_tensors="pt",
#         padding=True,
#         truncation=True
#     )
    
#     return inputs


In [339]:
# val_data = FlickrDataset(final_val_pairs)

# val_loader = DataLoader(
#     val_data,
#     batch_size=32,
#     shuffle=False,
#     num_workers=0,
#     collate_fn=collate_fn
# )

In [340]:
print(type(dataset[0]["image"]))

<class 'PIL.JpegImagePlugin.JpegImageFile'>


In [341]:
# BASE_PATH = "/full/path/to/flickr8k/images"

In [342]:
# from PIL import Image
# import os
# import requests
# from io import BytesIO

# def prepare_images(image_list):
#     processed = []

#     for img in image_list:

#         # ✅ Case 1: already PIL
#         if isinstance(img, Image.Image):
#             processed.append(img.convert("RGB"))

#         # ✅ Case 2: local file path
#         elif isinstance(img, str) and os.path.exists(img):
#             processed.append(Image.open(img).convert("RGB"))

#         # ✅ Case 3: URL
#         elif isinstance(img, str) and img.startswith("http"):
#             response = requests.get(img)
#             processed.append(Image.open(BytesIO(response.content)).convert("RGB"))

#         elif isinstance(img, str):
#             full_path = os.path.join(BASE_PATH, img)
#             processed.append(Image.open(full_path).convert("RGB"))

#         # ❌ Unknown type
#         else:
#             print("❌ Problematic item:", img)
#             print("Type:", type(img))
#             raise ValueError("Unsupported image format")


#     return processed

In [343]:
print(type(image_list[0]))
print(image_list[0])

<class 'str'>
image


In [344]:
image_list = [item["image"].convert("RGB") for item in val_dataset]
caption_list = [item["caption"] for item in val_dataset]

In [345]:
def encode_images(images, batch_size=32):
    all_features = []

    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]

        inputs = processor(
            images=batch,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            feats = model.get_image_features(**inputs)

        all_features.append(feats.cpu())

    return torch.cat(all_features, dim=0)

In [346]:
def encode_texts(texts, batch_size=32):
    all_features = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = processor(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            feats = model.get_text_features(**inputs)

        all_features.append(feats.cpu())

    return torch.cat(all_features, dim=0)

In [347]:
print("Encoding images...")
image_embeds = encode_images(image_list)

print("Encoding captions...")
text_embeds = encode_texts(caption_list)

print("Shapes:", image_embeds.shape, text_embeds.shape)

Encoding images...
Encoding captions...
Shapes: torch.Size([20, 512]) torch.Size([20, 512])


In [348]:
print(dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x1280 at 0x33E48BB20>, 'caption': 'a yellow and white cartoon character with a red eye'}


In [349]:
tau = 0.07
similarity = (text_embeds @ image_embeds.T) / tau

In [350]:
def compute_recall(similarity, k_list=[1,5,10]):
    recalls = {k: 0 for k in k_list}
    n = similarity.shape[0]

    for i in range(n):
        sims = similarity[i]
        top_k = torch.topk(sims, max(k_list)).indices

        for k in k_list:
            if i in top_k[:k]:
                recalls[k] += 1

    for k in recalls:
        recalls[k] /= n

    return recalls

In [351]:
recall = compute_recall(similarity)

print("MSCOCO Results:")
print(f"R@1: {recall[1]:.4f}")
print(f"R@5: {recall[5]:.4f}")
print(f"R@10: {recall[10]:.4f}")

MSCOCO Results:
R@1: 0.9500
R@5: 1.0000
R@10: 1.0000


In [352]:
def mean_similarity(image_embeds, text_embeds):
    sims = []

    for i in range(len(text_embeds)):
        sim = (text_embeds[i] @ image_embeds[i]).item()
        sims.append(sim)

    return np.mean(sims)

In [353]:
mean_sim = mean_similarity(image_embeds, text_embeds)
print("Mean similarity:", mean_sim)

Mean similarity: 22.494792413711547


In [354]:
def low_rank_failures(similarity, top_n=5):
    failures = []

    for i in range(len(similarity)):
        sims = similarity[i]

        ranking = torch.argsort(sims, descending=True)
        rank = (ranking == i).nonzero(as_tuple=True)[0].item()

        if rank > 50:
            failures.append((i, rank))

        if len(failures) >= top_n:
            break

    return failures

In [355]:
def wrong_top(similarity, top_n=5):
    failures = []

    for i in range(len(similarity)):
        sims = similarity[i]
        top1 = torch.argmax(sims).item()

        if top1 != i:
            failures.append((i, top1))

        if len(failures) >= top_n:
            break

    return failures

In [356]:
print("Low-rank failures:", low_rank_failures(similarity))
print("Wrong top predictions:", wrong_top(similarity))

Low-rank failures: []
Wrong top predictions: [(4, 18)]


In [357]:
flickr_results = {
    1: 0.54,   # replace with your actual values
    5: 0.79,
    10: 0.86
}

In [358]:
def compute_drop(flickr_results, ood_results):
    drop = {}
    
    for k in flickr_results:
        drop[k] = flickr_results[k] - ood_results[k]
    
    return drop

In [359]:
drop = compute_drop(flickr_results, recall)

print("Accuracy Drop:")
print(f"ΔR@1:  {drop[1]:.4f}")
print(f"ΔR@5:  {drop[5]:.4f}")
print(f"ΔR@10: {drop[10]:.4f}")

Accuracy Drop:
ΔR@1:  -0.4100
ΔR@5:  -0.2100
ΔR@10: -0.1400


In [360]:
# from datasets import Dataset

# # Set seed for reproducibility
# seed = 42

# # Step 1: Shuffle full dataset
# shuffled_dataset = full_dataset.shuffle(seed=seed)

# # Step 2: Select splits
# train_dataset = shuffled_dataset.select(range(100))
# # val_dataset = shuffled_dataset.select(range(20000, 23000))

# # Print info

print("Training size:", len(train_dataset))

Training size: 80


In [361]:
from collections import defaultdict
import random

train_image_to_captions = defaultdict(list)

# Step 1: Build mapping
for item in train_dataset:
    image = item["image"]
    captions = item["caption"]
    
    for caption in captions:
        train_image_to_captions[id(image)].append((image, caption))

print("Unique images:", len(train_image_to_captions))


# Step 2: Select 1 caption per image and store rest separately
train_unique_image_pairs = []
train_remaining_pairs = []

for image_id, captions_list in train_image_to_captions.items():
    
    # Randomly choose one caption for uniqueness
    chosen_pair = random.choice(captions_list)
    train_unique_image_pairs.append(chosen_pair)
    
    # Add the remaining captions directly (no comparison needed)
    for pair in captions_list:
        if pair != chosen_pair:
            train_remaining_pairs.append(pair)

print("Unique pairs:", len(train_unique_image_pairs))
print("Remaining pairs:", len(train_remaining_pairs))

Unique images: 80
Unique pairs: 80
Remaining pairs: 3590


In [362]:
final_train_pairs=train_unique_image_pairs 
random.shuffle(final_train_pairs)


print("Final training dataset size:", len(final_train_pairs))


Final training dataset size: 80


In [363]:
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


In [364]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [365]:
import torch
import torch.nn as nn
import numpy as np

logit_scale = nn.Parameter(torch.tensor(np.log(1/0.07), dtype=torch.float32))
logit_scale = logit_scale.to(device)

In [366]:
clean_pairs = []

for img, cap in final_train_pairs:
# for img, cap in train_pairs:
    try:
        if img is None:
            continue
            
        # ensure PIL image RGB
        img = img.convert("RGB")
        
        # ensure caption string
        cap = str(cap)
        
        clean_pairs.append((img, cap))
        
    except:
        continue

len(clean_pairs)


80

In [367]:
from torch.utils.data import Dataset

In [368]:
class FlickrDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        image, caption = self.pairs[idx]
        return image, caption


In [369]:
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def collate_fn(batch):
    images, captions = zip(*batch)
    
    inputs = processor(
        text=list(captions),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    
    return inputs


In [370]:
from torch.utils.data import DataLoader

In [371]:
train_pairs = final_train_pairs

train_data = FlickrDataset(train_pairs)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

In [372]:
batch = next(iter(train_loader))

for key in batch:
    print(key, batch[key].shape)


input_ids torch.Size([32, 3])
attention_mask torch.Size([32, 3])
pixel_values torch.Size([32, 3, 224, 224])


In [373]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

batch = {k: v.to(device) for k, v in batch.items()}


In [374]:
import torch
from transformers import CLIPModel

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model = model.to(device)

model.train()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [375]:
batch = next(iter(train_loader))
print(batch.keys())

dict_keys(['input_ids', 'attention_mask', 'pixel_values'])


In [376]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-6)


from torch.optim import AdamW

logit_scale = logit_scale.detach().requires_grad_()
optimizer = AdamW(
    list(model.parameters()) + [logit_scale],
    lr=2e-6
)

# from torch.optim import AdamW

# # make sure logit_scale is a leaf parameter
# logit_scale = logit_scale.detach().requires_grad_()

# optimizer = AdamW([
    
#     # Vision encoder (image model) → smaller LR
#     {"params": model.vision_model.parameters(), "lr": 1e-6},
    
#     # Text encoder → higher LR
#     {"params": model.text_model.parameters(), "lr": 5e-6},
    
#     # Projection layers (important for alignment)
#     {"params": model.visual_projection.parameters(), "lr": 5e-6},
#     {"params": model.text_projection.parameters(), "lr": 5e-6},
    
#     # Learned temperature
#     {"params": [logit_scale], "lr": 5e-6}
# ])


In [377]:
epochs = 1

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch in train_loader:
        
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        
        image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
        text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
        
        similarity = logit_scale.exp() * (image_embeds @ text_embeds.T)
        
        labels = torch.arange(similarity.size(0)).to(device)
        
        loss_i = torch.nn.functional.cross_entropy(similarity, labels)
        loss_t = torch.nn.functional.cross_entropy(similarity.T, labels)
        loss = (loss_i + loss_t) / 2
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
           
    avg_train_loss = total_loss / num_batches
    
    # validation loss
    # avg_val_loss = compute_validation_loss(model, val_loader, logit_scale)
    # # avg_val_loss = val_loss / len(val_loader)
    
    # # validation recall
    # val_recall = compute_recall(model, val_loader)
    
    print(f"\nEpoch {epoch+1}")
    print("Average Train Loss:", avg_train_loss)
    # print("Average Validation Loss:", avg_val_loss)
    # print("Validation Recall@10:", val_recall)
    print("Temperature:", 1 / logit_scale.exp().item())

#     # ----- EARLY STOPPING LOGIC -----
#     if val_recall > best_val_recall:
#         best_val_recall = val_recall
#         patience_counter = 0
        
#         # Save best model weights
#         best_model_state = model.state_dict()
#         print("New best model saved.")
        
#     else:
#         patience_counter += 1
#         print(f"No improvement. Patience: {patience_counter}/{patience}")
        
#     if patience_counter >= patience:
#         print("\nEarly stopping triggered.")
#         break

# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print("Best model restored.")


Epoch 1
Average Train Loss: 3.253299077351888
Temperature: 0.07000040254824387


In [378]:
len(final_val_pairs)

20

In [379]:
val_data = FlickrDataset(final_val_pairs)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

In [380]:
model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [381]:
# Extract and pair images and captions from val_dataset
image_list = [item["image"].convert("RGB") for item in val_dataset]
caption_list = [item["caption"] for item in val_dataset]

# Create list of tuples
final_val_pairs = list(zip(image_list, caption_list))

# Now create the DataLoader
val_data = FlickrDataset(final_val_pairs)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

In [382]:
all_image_embeds = []
all_text_embeds = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        
        image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
        text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
        
        all_image_embeds.append(image_embeds)
        all_text_embeds.append(text_embeds)

all_image_embeds = torch.cat(all_image_embeds)
all_text_embeds = torch.cat(all_text_embeds)

print(all_image_embeds.shape)
print(all_text_embeds.shape)

torch.Size([20, 512])
torch.Size([20, 512])


In [383]:
similarity = all_text_embeds @ all_image_embeds.T
temperature = 0.07
similarity = similarity / temperature

In [384]:
import numpy as np

similarity_np = similarity.cpu().numpy()

recall_at_1 = 0
recall_at_5 = 0
recall_at_10 = 0

for i in range(len(similarity_np)):
    
    # sort indices in descending similarity
    ranked_indices = np.argsort(-similarity_np[i])
    
    if i in ranked_indices[:1]:
        recall_at_1 += 1
        
    if i in ranked_indices[:5]:
        recall_at_5 += 1
        
    if i in ranked_indices[:10]:
        recall_at_10 += 1

N = len(similarity_np)

print("Recall@1:", recall_at_1 / N)
print("Recall@5:", recall_at_5 / N)
print("Recall@10:", recall_at_10 / N)

Recall@1: 1.0
Recall@5: 1.0
Recall@10: 1.0


In [385]:
# Before fine-tuning (zero-shot)
before_results = {
    1: recall[1],
    5: recall[5],
    10: recall[10]
}

# After fine-tuning (adapted model)
after_results = {
    1: recall_at_1 / N,
    5: recall_at_5 / N,
    10: recall_at_10 / N
}

In [386]:
def compute_adaptation_gain(before, after):
    return {k: after[k] - before[k] for k in before}

In [387]:
gain = compute_adaptation_gain(before_results, after_results)

In [388]:
print("\n===== ADAPTATION GAIN =====")

for k in [1, 5, 10]:
    print(f"R@{k}: Before={before_results[k]:.4f} | After={after_results[k]:.4f} | Gain={gain[k]:+.4f}")


===== ADAPTATION GAIN =====
R@1: Before=0.9500 | After=1.0000 | Gain=+0.0500
R@5: Before=1.0000 | After=1.0000 | Gain=+0.0000
R@10: Before=1.0000 | After=1.0000 | Gain=+0.0000
